# Data Quality Gates — `customers_raw`

Runs the five Pandera data-quality gates from `src/telco_churn/data/` against the live
`customers_raw` Postgres table and renders any failures for interactive inspection.

| Gate | Check | Severity |
|---|---|---|
| 1 | Schema — column presence, types, value ranges, categoricals | ERROR |
| 2 | Duplicate `customerid` values | ERROR |
| 3 | Binary churn labels, no missing values | ERROR |
| 4 | Unexpected NULL `totalcharges` (non-zero tenure) | WARNING |
| 5 | Row count ≥ 1 000; critical-column null rates ≤ 5 % | ERROR / WARNING |

**Requires:** `docker compose --profile infra up -d` and `POSTGRES_URL` in `.env`.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from telco_churn.data.validate import (
    clean_dataframe,
    validate_clean,
    validate_raw,
)
from telco_churn.utils.db import get_engine
from telco_churn.utils.logging import configure_logging

load_dotenv()
configure_logging()

REPORTS_DIR = Path("reports/validation")

## 1. Load data from Postgres

In [2]:
engine = get_engine()
df_raw = pd.read_sql("SELECT * FROM customers_raw", engine)
print(f"Loaded {len(df_raw):,} rows × {len(df_raw.columns)} columns")
df_raw.head()

Loaded 7,043 rows × 21 columns


,customerid,gender,seniorcitizen,has_partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract_type,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


## 2. Run the five data-quality gates

In [3]:
result = validate_raw(df_raw, strict=False, reports_dir=REPORTS_DIR)

## 3. Gate results

In [4]:
status = "PASS — pipeline can proceed" if result.can_proceed else "FAIL — blocking errors found"
print(f"Overall: {status}")
print(f"Errors  : {len(result.errors)}")
print(f"Warnings: {len(result.warnings)}")

gate_summary = pd.DataFrame(
    [
        {
            "gate": c.name,
            "passed": c.passed,
            "severity": str(c.failure_severity),
            "affected_rows": c.affected_rows,
            "message": c.message,
        }
        for c in result.checks
    ]
)
display(gate_summary)

Overall: PASS — pipeline can proceed
Errors  : 0
Warnings: 0


,gate,passed,severity,affected_rows,message
0,schema,True,ERROR,0,Schema validation passed.
1,duplicate_ids,True,ERROR,0,No duplicate customerid values.
2,churn_labels,True,ERROR,0,"Churn labels are valid (binary, no missing val..."
3,totalcharges_unexpected_nulls,True,WARNING,0,No unexpected NULL totalcharges values.
4,row_count,True,WARNING,0,Row count 7043 meets the minimum threshold of ...
5,null_rate_churn,True,WARNING,0,'churn' null rate 0.0% is within threshold.
6,null_rate_customerid,True,WARNING,0,'customerid' null rate 0.0% is within threshold.
7,null_rate_tenure,True,WARNING,0,'tenure' null rate 0.0% is within threshold.


### Failure details

The cells below mirror the `summary.csv` and `schema_failures.csv` written to
`reports/validation/` by `validate_raw`. They render as `None` when all gates pass.

In [5]:
# summary.csv equivalent — failing checks only
failing = [c for c in result.checks if not c.passed]

if not failing:
    print("summary.csv: no failing checks — file not written.")
else:
    summary_csv = pd.DataFrame(
        [
            {
                "check": c.name,
                "failure_severity": str(c.failure_severity),
                "message": c.message,
                "affected_rows": c.affected_rows,
            }
            for c in failing
        ]
    )
    print("summary.csv")
    display(summary_csv)

summary.csv: no failing checks — file not written.


In [6]:
# schema_failures.csv equivalent — row-level pandera failure cases
schema_check = next((c for c in result.checks if c.name == "schema"), None)

if schema_check is None or schema_check.passed:
    print("schema_failures.csv: schema check passed — file not written.")
else:
    print("schema_failures.csv")
    display(schema_check.detail)

schema_failures.csv: schema check passed — file not written.


## 4. Impute `totalcharges` and re-validate

`clean_dataframe` fills NULL `totalcharges` for the 11 zero-tenure customers with the
column median. `validate_clean` re-runs all gates using `CleanedSchema`, which requires
`totalcharges` to be non-null.

In [7]:
df_clean = clean_dataframe(df_raw)

nulls_before = int(df_raw["totalcharges"].isna().sum())
nulls_after = int(df_clean["totalcharges"].isna().sum())
print(f"NULL totalcharges: {nulls_before} → {nulls_after} after imputation")

result_clean = validate_clean(df_clean, strict=False, reports_dir=REPORTS_DIR)

status_clean = "PASS" if result_clean.can_proceed else "FAIL"
print(f"Post-imputation validation: {status_clean}")
print(f"Errors: {len(result_clean.errors)}  Warnings: {len(result_clean.warnings)}")

NULL totalcharges: 11 → 0 after imputation
Post-imputation validation: PASS
Errors: 0  Warnings: 0


---

## Summary

This notebook validates the raw `customers_raw` table against five automated quality gates and confirms post-imputation cleanliness. All gates pass on the IBM Telco dataset.

For a narrative summary of the data quality findings — including missing value treatment, outlier analysis, and categorical cardinality — see **[§2 Data Quality & Missing Values](../ANALYSIS.md#2-data-quality--missing-values)** in `ANALYSIS.md`.